# Final project

Team1: Karl Prokop and Amanda Christianson

# Checkpoint 1: Sentinel-2 data selection and retrieval 

### Importing libraries and configuration of OAuth2 Client Credentials

In [1]:
# Importing libraries
import requests
import os
import json
from datetime import datetime
import zipfile
from pathlib import Path

# For visualization later
import matplotlib.pyplot as plt

In [2]:
# Configuration of credentials

# Direct assignment
COPERNICUS_CLIENT_ID = os.getenv('COPERNICUS_CLIENT_ID', 'sh-c5dcc309-63e8-491b-8c97-47925cbe91ea')
COPERNICUS_CLIENT_SECRET = os.getenv('COPERNICUS_CLIENT_SECRET', 'uZ0NKF3IlVgFdGQUbSOYlLDrdbjudxtL')

# Karl's key
# sh-488cead5-2fcf-4384-ae68-0929267aa550
# 3fPSrDkk2UdHWozCmIFnGHxgAnyZc6jj

# Amanda's key
# sh-c5dcc309-63e8-491b-8c97-47925cbe91ea
# uZ0NKF3IlVgFdGQUbSOYlLDrdbjudxtL

# Copernicus Dataspace API endpoints
AUTH_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SEARCH_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
DOWNLOAD_URL = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"

print("✓ Credentials configured")

✓ Credentials configured


In [3]:
def get_access_token(client_id, client_secret):
    """
    Get OAuth2 access token from Copernicus Dataspace using Client Credentials flow.
    
    This is the recommended method for server-to-server authentication and HPC jobs.
    
    Parameters:
    -----------
    client_id : str
        OAuth2 Client ID (starts with 'sh-')
    client_secret : str
        OAuth2 Client Secret
    
    Returns:
    --------
    str : Access token if successful, None otherwise
    """
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }
    
    try:
        response = requests.post(AUTH_URL, data=data, timeout=30)
        response.raise_for_status()
        token_data = response.json()
        
        # Extract token and expiration
        access_token = token_data["access_token"]
        expires_in = token_data.get("expires_in", 3600)
        
        print(f"✓ Token obtained (valid for {expires_in//60} minutes)")
        return access_token
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            print("❌ Authentication failed: Invalid credentials")
            print("   ✗ Check your CLIENT_ID and CLIENT_SECRET")
            print("   ✗ CLIENT_ID should start with 'sh-'")
        else:
            print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
        return None
        
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        return None

# Get access token
print("Authenticating with Copernicus Dataspace...")
access_token = get_access_token(COPERNICUS_CLIENT_ID, COPERNICUS_CLIENT_SECRET)

if access_token:
    print("✓ Successfully authenticated")
    headers = {"Authorization": f"Bearer {access_token}"}
else:
    print("❌ Authentication failed.")
    print("\nTroubleshooting:")
    print("1. Check environment variables are set:")
    print(f"   COPERNICUS_CLIENT_ID = {COPERNICUS_CLIENT_ID[:15]}...")
    print(f"   COPERNICUS_CLIENT_SECRET = {COPERNICUS_CLIENT_SECRET[:15]}...")
    print("2. Verify credentials in Copernicus Dashboard")
    print("3. See COPERNICUS_SETUP.md for detailed instructions")
    headers = None

Authenticating with Copernicus Dataspace...
✓ Token obtained (valid for 30 minutes)
✓ Successfully authenticated


### Define Search Parameters (region of interest and date range)

In [86]:
# Define your Region of Interest (ROI) as a bounding box
# Format: POLYGON((lon lat, lon lat, ...))
# Note: You will be selecting an MGRS tile within this region

# Bounding box coordinates [min_lon, min_lat, max_lon, max_lat]
# Middle of Sweden (Focus Dalarna but covering Uppsala to Umeå), Sweden
min_lon, min_lat = 12.5, 60
max_lon, max_lat = 17.5, 63.5

# Create WKT POLYGON for API query
roi_polygon = f"POLYGON(({min_lon} {min_lat},{max_lon} {min_lat},{max_lon} {max_lat},{min_lon} {max_lat},{min_lon} {min_lat}))"

# Define date range
start_date = '2018-03-01T00:00:00.000Z'
end_date = '2018-10-31T23:59:59.999Z'

# Maximum cloud cover percentage (30% to get good data availability)
max_cloud_cover = 30

print(f"Search Parameters:")
print(f"  Region: Dalarna Sweden (Central Europe with CORINE coverage)")
print(f"  Bounding Box: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")
print(f"  Date Range: {start_date[:10]} to {end_date[:10]}")
print(f"  Max Cloud Cover: {max_cloud_cover}%")
print(f"\nNote: Sentinel-2 divides the globe into MGRS tiles (100×100 km each).")
print(f"Your search will find all tiles intersecting this region.")

Search Parameters:
  Region: Dalarna Sweden (Central Europe with CORINE coverage)
  Bounding Box: (12.5, 60) to (17.5, 63.5)
  Date Range: 2018-03-01 to 2018-10-31
  Max Cloud Cover: 30%

Note: Sentinel-2 divides the globe into MGRS tiles (100×100 km each).
Your search will find all tiles intersecting this region.


## Search Sentinel-2 Collection

Query the Copernicus Dataspace catalog for Sentinel-2 Level 2A imagery matching our criteria.

In [87]:
def search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover):
    """
    Search for Sentinel-2 L2A products in Copernicus Dataspace
    """
    # Build OData filter query
    filters = [
        f"Collection/Name eq 'SENTINEL-2'",
        f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A')",
        f"ContentDate/Start gt {start_date}",
        f"ContentDate/Start lt {end_date}",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{roi_polygon}')",
        f"Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt {max_cloud_cover})"
    ]
    
    filter_query = " and ".join(filters)
    
    params = {
        "$filter": filter_query,
        "$orderby": "ContentDate/Start asc",
        "$top": 1000  # Increased to get full date range across all tiles
    }
    
    try:
        response = requests.get(SEARCH_URL, params=params, timeout=60)
        response.raise_for_status()
        results = response.json()
        return results.get('value', [])
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return []

print("✓ Search function defined")

✓ Search function defined


In [88]:
print("Searching for Sentinel-2 products...")
products = search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover)

print(f"\n✓ Found {len(products)} Sentinel-2 L2A products")
print(f"✓ All products have <{max_cloud_cover}% cloud cover (filtered server-side)")
print(f"\nThese products span multiple MGRS tiles over your region.")
print(f"Select one MGRS tile and download ~4 acquisitions.\n")
print(f"First 5 products:")
for i, product in enumerate(products[:5]):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    
    print(f"  {i+1}. {name}")
    print(f"     Date: {date}, Size: {size:.2f} GB")

Searching for Sentinel-2 products...

✓ Found 1000 Sentinel-2 L2A products
✓ All products have <30% cloud cover (filtered server-side)

These products span multiple MGRS tiles over your region.
Select one MGRS tile and download ~4 acquisitions.

First 5 products:
  1. S2B_MSIL2A_20180301T103019_N0500_R108_T33VUK_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.20 GB
  2. S2B_MSIL2A_20180301T103019_N0500_R108_T32VPQ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.12 GB
  3. S2B_MSIL2A_20180301T103019_N0500_R108_T33VWJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.18 GB
  4. S2B_MSIL2A_20180301T103019_N0500_R108_T33VXL_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.17 GB
  5. S2B_MSIL2A_20180301T103019_N0500_R108_T33VVJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.16 GB


## Group Products by MGRS Tile

Organize products by MGRS tile to ensure we download multiple acquisitions of the same tile.

In [89]:
# Group products by MGRS tile
from collections import defaultdict

tiles = defaultdict(list)
for product in products:
    product_name = product.get('Name', '')
    # Extract MGRS tile from product name (e.g., T32UPD from S2A_MSIL2A_..._T32UPD_...)
    tile_id = product_name.split('_')[5] if len(product_name.split('_')) > 5 else 'Unknown'
    tiles[tile_id].append(product)

# Display available tiles and their acquisition counts
print("Available MGRS Tiles and Acquisition Counts:")
print("=" * 50)
for tile_id, tile_products in sorted(tiles.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\nTile {tile_id}: {len(tile_products)} acquisitions")
    for i, product in enumerate(tile_products[:5]):  # Show first 5
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. {date} - {size:.2f} GB")
    if len(tile_products) > 5:
        print(f"  ... and {len(tile_products) - 5} more")

print("\n" + "=" * 50)
print(f"\nRecommendation: Choose a tile with 4+ acquisitions for training data diversity.")

Available MGRS Tiles and Acquisition Counts:

Tile T33VWH: 58 acquisitions
  1. 2018-03-14 - 0.10 GB
  2. 2018-03-15 - 0.17 GB
  3. 2018-03-16 - 1.16 GB
  4. 2018-03-18 - 1.18 GB
  5. 2018-03-20 - 0.18 GB
  ... and 53 more

Tile T34VCN: 48 acquisitions
  1. 2018-03-18 - 0.95 GB
  2. 2018-03-20 - 0.94 GB
  3. 2018-03-21 - 0.12 GB
  4. 2018-03-25 - 0.84 GB
  5. 2018-03-26 - 0.10 GB
  ... and 43 more

Tile T34VCP: 46 acquisitions
  1. 2018-03-01 - 0.51 GB
  2. 2018-03-11 - 0.61 GB
  3. 2018-03-20 - 0.79 GB
  4. 2018-03-25 - 0.70 GB
  5. 2018-03-26 - 0.46 GB
  ... and 41 more

Tile T33VXJ: 46 acquisitions
  1. 2018-03-01 - 0.73 GB
  2. 2018-03-11 - 0.82 GB
  3. 2018-03-20 - 0.69 GB
  4. 2018-03-21 - 0.74 GB
  5. 2018-03-25 - 0.60 GB
  ... and 41 more

Tile T33VXH: 45 acquisitions
  1. 2018-03-18 - 1.08 GB
  2. 2018-03-20 - 1.02 GB
  3. 2018-03-21 - 0.43 GB
  4. 2018-03-25 - 0.98 GB
  5. 2018-03-28 - 1.01 GB
  ... and 40 more

Tile T33VWG: 43 acquisitions
  1. 2018-03-20 - 0.53 GB
  2. 2018

### Alt 1: Automatically pick 4 equally spaced acquisitions for a specified tile

In [17]:
# Select a tile to work with (choose the one with most acquisitions, or specify manually)
# Option 1: Automatic - select tile with most acquisitions
#selected_tile = max(tiles.items(), key=lambda x: len(x[1]))[0] if tiles else None

# Option 2: Manual selection - uncomment and specify tile ID
selected_tile = "T33VWH"  # Replace with your chosen tile

if selected_tile:
    tile_products = tiles[selected_tile]
    num_acquisitions = len(tile_products)
    
    print(f"Selected MGRS Tile: {selected_tile}")
    print(f"Total acquisitions available: {num_acquisitions}")
    
    # Select 4 evenly spaced acquisitions for temporal diversity
    num_to_select = 4
    if num_acquisitions >= num_to_select:
        # Calculate indices for evenly spaced selection
        indices = [int(i * (num_acquisitions - 1) / (num_to_select - 1)) for i in range(num_to_select)]
        selected_products = [tile_products[i] for i in indices]
    else:
        # If fewer than 4 acquisitions, use all of them
        selected_products = tile_products
        indices = list(range(len(tile_products)))
    
    print(f"\nSelected {len(selected_products)} evenly-spaced acquisitions for temporal diversity:")
    print("=" * 70)
    for i, (idx, product) in enumerate(zip(indices, selected_products)):
        name = product.get('Name', 'Unknown')
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. [{idx+1}/{num_acquisitions}] {date} - {size:.2f} GB")
        print(f"      {name}")
    print("=" * 70)
    print("\nThese acquisitions span the full date range for better training data diversity.")
else:
    print("❌ No tiles found. Adjust your search parameters.")
    selected_products = []

Selected MGRS Tile: T33VWH
Total acquisitions available: 58

Selected 4 evenly-spaced acquisitions for temporal diversity:
  1. [1/58] 2018-03-14 - 0.10 GB
      S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE
  2. [20/58] 2018-05-09 - 0.15 GB
      S2A_MSIL2A_20180509T101031_N0500_R022_T33VWH_20230825T064921.SAFE
  3. [39/58] 2018-06-12 - 0.07 GB
      S2B_MSIL2A_20180612T104019_N0500_R008_T33VWH_20230716T012327.SAFE
  4. [58/58] 2018-08-11 - 0.08 GB
      S2B_MSIL2A_20180811T104019_N0500_R008_T33VWH_20230711T150315.SAFE

These acquisitions span the full date range for better training data diversity.


### Alt 2: Manually pick acquisitions by index

#### List all acquisitions for given tile

In [71]:
selected_tile = "T33VWH"  # Replace with your chosen tile

if selected_tile:
    tile_products = tiles[selected_tile]
    num_acquisitions = len(tile_products)

    print(f"Selected MGRS Tile: {selected_tile}")
    print(f"Total acquisitions available: {num_acquisitions}")

    # Print ALL acquisitions
    print("\nAll acquisitions:")
    print("=" * 70)
    for i, product in enumerate(tile_products, start=1):
        name = product.get('Name', 'Unknown')
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"{i:3d}. {date} - {size:.2f} GB")
        print(f"     {name}")
    print("=" * 70)

Selected MGRS Tile: T33VWH
Total acquisitions available: 58

All acquisitions:
  1. 2018-03-14 - 0.10 GB
     S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE
  2. 2018-03-15 - 0.17 GB
     S2B_MSIL2A_20180315T101019_N0500_R022_T33VWH_20230908T210616.SAFE
  3. 2018-03-16 - 1.16 GB
     S2A_MSIL2A_20180316T103021_N0500_R108_T33VWH_20230727T202428.SAFE
  4. 2018-03-18 - 1.18 GB
     S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE
  5. 2018-03-20 - 0.18 GB
     S2A_MSIL2A_20180320T101021_N0500_R022_T33VWH_20230903T133606.SAFE
  6. 2018-03-25 - 0.17 GB
     S2B_MSIL2A_20180325T101019_N0500_R022_T33VWH_20230828T070713.SAFE
  7. 2018-03-28 - 1.16 GB
     S2B_MSIL2A_20180328T102019_N0500_R065_T33VWH_20230829T155330.SAFE
  8. 2018-03-30 - 0.18 GB
     S2A_MSIL2A_20180330T101021_N0500_R022_T33VWH_20230828T061911.SAFE
  9. 2018-04-02 - 1.16 GB
     S2A_MSIL2A_20180402T102021_N0500_R065_T33VWH_20230726T115942.SAFE
 10. 2018-04-12 - 1.18 GB
     S2A_MSIL2A_201804

#### Pick four acquisitions

In [72]:
indices = [3,15,34,56] # List of indices for picked acquisitions
selected_products = [tile_products[i] for i in indices]

# Advice for picking good acquisitions: 
# Pick the ones with large file sizes (over 1 GB), they tend to not have black areas.
# List of good indicies for tile T33VWH: 3 (2018-03-18), 15 (2018-04-22), 34 (2018-06-04) , 56 (2018-08-08)

print("List of chosen acquisitions")
print("=" * 70)
for i, (idx, product) in enumerate(zip(indices, selected_products)):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    print(f"  {i+1}. [{idx+1}/{num_acquisitions}] {date} - {size:.2f} GB")
    print(f"      {name}")
print("=" * 70)

List of chosen acquisitions
  1. [4/58] 2018-03-18 - 1.18 GB
      S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE
  2. [16/58] 2018-04-22 - 1.18 GB
      S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE
  3. [35/58] 2018-06-04 - 1.06 GB
      S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511.SAFE
  4. [57/58] 2018-08-08 - 1.03 GB
      S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE


## Download Sentinel-2 Products

### Download function

In [73]:
def download_product(product, output_dir, access_token):
    """
    Download a Sentinel-2 product from Copernicus Dataspace
    
    Parameters:
    -----------
    product : dict
        Product metadata from search results
    output_dir : str
        Directory to save downloaded file
    access_token : str
        OAuth2 access token
    
    Returns:
    --------
    str : Path to downloaded file, or None if failed
    """
    product_id = product['Id']
    product_name = product['Name']
    
    # Create download directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Output file path
    output_file = os.path.join(output_dir, f"{product_name}.zip")
    
    # Check if already downloaded
    if os.path.exists(output_file):
        print(f"⚠ File already exists: {product_name}.zip")
        return output_file
    
    # Build download URL
    download_url = f"{DOWNLOAD_URL}({product_id})/$value"
    
    headers = {"Authorization": f"Bearer {access_token}"}
    
    try:
        print(f"Downloading: {product_name}")
        print(f"  Size: {product['ContentLength'] / (1024**3):.2f} GB")
        
        # Stream download with progress
        with requests.get(download_url, headers=headers, stream=True, timeout=300) as response:
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            block_size = 8192
            downloaded = 0
            
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=block_size):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        # Print progress every 100 MB
                        if downloaded % (100 * 1024 * 1024) < block_size:
                            progress = (downloaded / total_size) * 100 if total_size > 0 else 0
                            print(f"  Progress: {progress:.1f}% ({downloaded / (1024**3):.2f} GB)")
        
        print(f"✓ Download complete: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        # Clean up partial download
        if os.path.exists(output_file):
            os.remove(output_file)
        return None

print("✓ Download function defined")

✓ Download function defined


### Download A Selected Acquisition to scratch directory for team1

In [44]:
# Define download directory (modify for your HPC storage)
download_dir = f"/p/scratch/training2600/team1/data/T33VWH"

# Download the selected product
if products and access_token:
    downloaded_file = download_product(products[0], download_dir, access_token)
    if downloaded_file:
        print(f"\n✓ Product saved to: {downloaded_file}")
else:
    print("❌ Cannot download: No products found or authentication failed")

Downloading: S2B_MSIL2A_20180301T103019_N0500_R108_T33VUK_20230731T095708.SAFE
  Size: 0.20 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(022bc2b8-9288-44b4-8627-b6f79a67f09a)/$value


### Download Multiple Acquisitions to scratch directory for team1

In [13]:
# Download the evenly-spaced acquisitions from the selected tile
download_dir = f"/p/scratch/training2600/team1/data"

if selected_tile and access_token and selected_products:
    print(f"Downloading {len(selected_products)} evenly-spaced acquisitions from tile {selected_tile}...")
    print("=" * 60)
    
    for i, product in enumerate(selected_products):
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        print(f"\n--- Downloading {i+1}/{len(selected_products)} ({date}) ---")
        download_product(product, download_dir, access_token)
    
    print("\n" + "=" * 60)
    print(f"✓ Downloaded {len(selected_products)} acquisitions from tile {selected_tile}")
    print(f"✓ Acquisitions are evenly spaced across the time range")
    print(f"✓ All files saved to: {download_dir}")
else:
    print("❌ Cannot download: No tile/products selected or authentication failed")


--- Downloading 1/4 (2018-03-14) ---
Downloading: S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE
  Size: 0.10 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(432e0e5d-85fa-4906-ac98-7008973d6db1)/$value

--- Downloading 2/4 (2018-05-09) ---
Downloading: S2A_MSIL2A_20180509T101031_N0500_R022_T33VWH_20230825T064921.SAFE
  Size: 0.15 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(2ff2313b-c53a-475a-9da0-f38157f8c61c)/$value

--- Downloading 3/4 (2018-06-12) ---
Downloading: S2B_MSIL2A_20180612T104019_N0500_R008_T33VWH_20230716T012327.SAFE
  Size: 0.07 GB
❌ Download failed: 401 Client Error: Unauthorized for url: https://zipper.dataspace.copernicus.eu/odata/v1/Products(77a9f78d-6b87-4f1c-b07b-9a40fa39945b)/$value

--- Downloading 4/4 (2018-08-11) ---
Downloading: S2B_MSIL2A_20180811T104019_N0500_R008_T33VWH_20230711T150315.SAFE
 

## Alternative Download: Session Based

In [74]:
download_dir = f"/p/scratch/training2600/team1/data"

def create_authenticated_session(username, password):
    """
    Create a session with authentication for downloads
    """
    session = requests.Session()
    
    # Get access token
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    
    response = session.post(AUTH_URL, data=data, timeout=30)
    response.raise_for_status()
    
    token = response.json()["access_token"]
    session.headers.update({"Authorization": f"Bearer {token}"})
    
    return session

def download_with_session(product, output_dir, session):
    """
    Download using authenticated session
    """
    product_id = product['Id']
    product_name = product['Name']
    output_file = os.path.join(output_dir, f"{product_name}.zip")
    
    if os.path.exists(output_file):
        print(f"⚠ File already exists: {product_name}.zip")
        return output_file
    
    download_url = f"{DOWNLOAD_URL}({product_id})/$value"
    
    try:
        print(f"Downloading: {product_name}")
        print(f"  Size: {product['ContentLength'] / (1024**3):.2f} GB")
        
        with session.get(download_url, stream=True, timeout=300) as response:
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            downloaded = 0
            
            os.makedirs(output_dir, exist_ok=True)
            
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        if downloaded % (100 * 1024 * 1024) < 8192:
                            progress = (downloaded / total_size) * 100 if total_size > 0 else 0
                            print(f"  Progress: {progress:.1f}%")
        
        print(f"✓ Download complete: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        if os.path.exists(output_file):
            os.remove(output_file)
        return None

# Usage:
COPERNICUS_USERNAME = "acc6@hi.is"  # Your login email
COPERNICUS_PASSWORD = "CutiePatootie5!" # Your login password

session = create_authenticated_session(COPERNICUS_USERNAME, COPERNICUS_PASSWORD)

print(f"Downloading {len(selected_products)} evenly-spaced acquisitions from tile {selected_tile}...")
print("=" * 60)

for i, product in enumerate(selected_products):
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    print(f"\n--- Downloading {i+1}/{len(selected_products)} ({date}) ---")
    download_with_session(product, download_dir, session)

print("\n" + "=" * 60)
print(f"✓ Downloaded {len(selected_products)} acquisitions from tile {selected_tile}")
print(f"✓ Acquisitions are evenly spaced across the time range")
print(f"✓ All files saved to: {download_dir}")


--- Downloading 1/4 (2018-03-18) ---
Downloading: S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE
  Size: 1.18 GB
  Progress: 8.3%
  Progress: 16.5%
  Progress: 24.8%
  Progress: 33.1%
  Progress: 41.3%
  Progress: 49.6%
  Progress: 57.9%
  Progress: 66.1%
  Progress: 74.4%
  Progress: 82.7%
  Progress: 90.9%
  Progress: 99.2%
✓ Download complete: /p/scratch/training2600/team1/data/T33VWH/S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE.zip

--- Downloading 2/4 (2018-04-22) ---
Downloading: S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE
  Size: 1.18 GB
  Progress: 8.2%
  Progress: 16.5%
  Progress: 24.7%
  Progress: 33.0%
  Progress: 41.2%
  Progress: 49.5%
  Progress: 57.7%
  Progress: 66.0%
  Progress: 74.2%
  Progress: 82.5%
  Progress: 90.7%
  Progress: 99.0%
✓ Download complete: /p/scratch/training2600/team1/data/T33VWH/S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE.zip

--- Downloading 3/4 (2018-06-04) ---


## Visualisation

In [81]:
import numpy as np
import glob

def extract_and_visualize_sentinel2(zip_path, output_dir=None):
    """
    Extract a Sentinel-2 ZIP file and create an RGB visualization.
    
    Parameters:
    -----------
    zip_path : str
        Path to the Sentinel-2 ZIP file
    output_dir : str, optional
        Directory to extract to. If None, extracts to same directory as ZIP.
    
    Returns:
    --------
    str : Path to extracted SAFE directory
    """
    import zipfile
    import rasterio
    from rasterio.plot import show
    
    if output_dir is None:
        output_dir = os.path.dirname(zip_path)
    
    # Extract ZIP file
    print(f"Extracting: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get the SAFE directory name (first item in the archive)
        safe_dir = zip_ref.namelist()[0].split('/')[0]
        safe_path = os.path.join(output_dir, safe_dir)
        
        if os.path.exists(safe_path):
            print(f"⚠ Already extracted: {safe_dir}")
        else:
            zip_ref.extractall(output_dir)
            print(f"✓ Extracted to: {safe_path}")
    
    return safe_path


def visualize_sentinel2_rgb(safe_path, figsize=(12, 12)):
    """
    Create an RGB visualization from Sentinel-2 SAFE directory.
    
    Uses bands B04 (Red), B03 (Green), B02 (Blue) at 10m resolution.
    
    Parameters:
    -----------
    safe_path : str
        Path to the extracted .SAFE directory
    figsize : tuple
        Figure size for the plot
    """
    import rasterio
    from rasterio.enums import Resampling
    
    # Find the 10m resolution bands (B02, B03, B04)
    # Path pattern: .SAFE/GRANULE/*/IMG_DATA/R10m/*_B0X_10m.jp2
    granule_path = os.path.join(safe_path, 'GRANULE')
    
    if not os.path.exists(granule_path):
        print(f"❌ GRANULE directory not found in {safe_path}")
        return
    
    # Get the tile subdirectory
    tile_dirs = [d for d in os.listdir(granule_path) if os.path.isdir(os.path.join(granule_path, d))]
    if not tile_dirs:
        print("❌ No tile directories found")
        return
    
    tile_dir = os.path.join(granule_path, tile_dirs[0])
    
    # Try R10m directory first (newer format), then IMG_DATA (older format)
    r10m_path = os.path.join(tile_dir, 'IMG_DATA', 'R10m')
    img_data_path = os.path.join(tile_dir, 'IMG_DATA')
    
    if os.path.exists(r10m_path):
        band_dir = r10m_path
        band_pattern = '*_B0{}_10m.jp2'
    else:
        band_dir = img_data_path
        band_pattern = '*_B0{}.jp2'
    
    # Find band files
    bands = {}
    for band_num in ['2', '3', '4']:
        pattern = os.path.join(band_dir, band_pattern.format(band_num))
        matches = glob.glob(pattern)
        if matches:
            bands[f'B0{band_num}'] = matches[0]
        else:
            # Try alternative pattern for older format
            alt_pattern = os.path.join(band_dir, f'*B0{band_num}*.jp2')
            alt_matches = glob.glob(alt_pattern)
            if alt_matches:
                bands[f'B0{band_num}'] = alt_matches[0]
    
    if len(bands) < 3:
        print(f"❌ Could not find all RGB bands. Found: {list(bands.keys())}")
        print(f"   Searched in: {band_dir}")
        return
    
    print(f"✓ Found bands: {list(bands.keys())}")
    
    # Read bands with downsampling for visualization (full resolution can be huge)
    downsample_factor = 10  # Read at 1/10 resolution for faster display
    
    rgb_bands = []
    for band_name in ['B04', 'B03', 'B02']:  # RGB order
        with rasterio.open(bands[band_name]) as src:
            # Calculate new dimensions
            new_height = src.height // downsample_factor
            new_width = src.width // downsample_factor
            
            # Read with resampling
            band_data = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.average
            )
            rgb_bands.append(band_data)
    
    # Stack into RGB array
    rgb = np.stack(rgb_bands, axis=-1)
    
    # Normalize for visualization (typical Sentinel-2 values range 0-10000)
    # Use percentile-based stretching for better visualization
    p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
    rgb_normalized = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # RGB composite
    axes[0].imshow(rgb_normalized)
    axes[0].set_title('Sentinel-2 RGB Composite (B4-B3-B2)', fontsize=12)
    axes[0].axis('off')
    
    # False color (NIR-Red-Green) if available
    # For now, show a histogram of the data
    axes[1].hist(rgb[:,:,0].flatten()[::100], bins=50, alpha=0.7, label='Red (B04)', color='red')
    axes[1].hist(rgb[:,:,1].flatten()[::100], bins=50, alpha=0.7, label='Green (B03)', color='green')
    axes[1].hist(rgb[:,:,2].flatten()[::100], bins=50, alpha=0.7, label='Blue (B02)', color='blue')
    axes[1].set_xlabel('Reflectance Value')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Band Value Distribution', fontsize=12)
    axes[1].legend()
    axes[1].set_xlim(0, 5000)
    
    plt.tight_layout()
    plt.savefig(os.path.join(os.environ['PROJECT_training2600'],  "team1/results/vis_S2.png"))
    
    # Print some metadata
    with rasterio.open(bands['B04']) as src:
        print(f"\nImage Metadata:")
        print(f"  Original size: {src.width} x {src.height} pixels")
        print(f"  Resolution: {src.res[0]}m x {src.res[1]}m")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
    
    return rgb_normalized

print("✓ Visualization functions defined")

✓ Visualization functions defined


In [82]:
# Visualize the first downloaded acquisition
# Adjust the path to match your download location

def is_valid_zipfile(filepath):
    """Check if a file is a valid ZIP file."""
    try:
        with zipfile.ZipFile(filepath, 'r') as zf:
            return zf.testzip() is None
    except (zipfile.BadZipFile, Exception):
        return False

if selected_products and access_token:
    # Get the first downloaded product
    first_product = selected_products[0] # Adjust to decide which product for vizualise
    product_name = first_product.get('Name', '')
    
    # Build possible paths - handle various naming conventions
    # The product name might already end with .SAFE
    base_name = product_name.rstrip('.SAFE') if product_name.endswith('.SAFE') else product_name
    
    possible_paths = [
        os.path.join(download_dir, f"{product_name}"),           # Direct name (if it's a directory)
        os.path.join(download_dir, f"{product_name}.SAFE"),      # With .SAFE suffix
        os.path.join(download_dir, f"{base_name}.SAFE"),         # Base name with .SAFE
        os.path.join(download_dir, f"{product_name}.zip"),       # With .zip suffix  
        os.path.join(download_dir, f"{base_name}.zip"),          # Base name with .zip
    ]
    
    safe_path = None
    zip_path = None
    
    # Find the first existing path
    for path in possible_paths:
        if os.path.exists(path):
            if os.path.isdir(path):
                # It's a directory (SAFE format)
                safe_path = path
                print(f"✓ Found SAFE directory: {os.path.basename(path)}")
                break
            elif path.endswith('.zip') and is_valid_zipfile(path):
                # It's a valid ZIP file
                zip_path = path
                print(f"✓ Found valid ZIP file: {os.path.basename(path)}")
                break
            elif path.endswith('.zip'):
                # File exists but is not a valid ZIP - might be misnamed SAFE dir
                print(f"⚠ Found {os.path.basename(path)} but it's not a valid ZIP file")
                print("  Checking if it might be a SAFE directory saved with wrong extension...")
                # Check if there's a SAFE directory with similar name
                continue
    
    # If we found a ZIP file, extract it
    if zip_path and not safe_path:
        print(f"\nVisualizing: {os.path.basename(zip_path)}")
        print("=" * 60)
        safe_path = extract_and_visualize_sentinel2(zip_path)
    
    # Visualize if we have a SAFE path
    if safe_path and os.path.isdir(safe_path):
        print(f"\nVisualizing: {os.path.basename(safe_path)}")
        print("=" * 60)
        
        # Create RGB visualization directly from SAFE directory
        print("\nCreating RGB visualization...")
        visualize_sentinel2_rgb(safe_path)
        
        print("\n" + "=" * 60)
        print("✓ Visualization complete!")
        print("  - RGB composite shows the natural color view")
        print("  - Histogram shows the distribution of reflectance values")
    else:
        # Last resort: try to find any .SAFE directory in download_dir
        safe_dirs = glob.glob(os.path.join(download_dir, "*.SAFE"))
        if safe_dirs:
            safe_path = safe_dirs[0]
            print(f"\nFound SAFE directory: {os.path.basename(safe_path)}")
            print("=" * 60)
            
            print("\nCreating RGB visualization...")
            visualize_sentinel2_rgb(safe_path)
            
            print("\n" + "=" * 60)
            print("✓ Visualization complete!")
        else:
            print(f"\n❌ No valid Sentinel-2 data found in: {download_dir}")
            print(f"\nSearched for:")
            for p in possible_paths[:3]:
                print(f"  - {p}")
            print("\nTip: Check what files exist in your download directory:")
            print(f"  ls -la {download_dir}")
else:
    print("❌ No products selected or authentication failed")
    print("   Run the search and download cells first.")

✓ Found valid ZIP file: S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE.zip

Visualizing: S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE.zip
Extracting: S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE.zip
✓ Extracted to: /p/scratch/training2600/team1/data/T33VWH/S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE

Visualizing: S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE

Creating RGB visualization...
✓ Found bands: ['B02', 'B03', 'B04']

Image Metadata:
  Original size: 10980 x 10980 pixels
  Resolution: 10.0m x 10.0m
  CRS: EPSG:32633
  Bounds: BoundingBox(left=499980.0, bottom=6690240.0, right=609780.0, top=6800040.0)

✓ Visualization complete!
  - RGB composite shows the natural color view
  - Histogram shows the distribution of reflectance values


# Checkpoint 2: Step 1

In [20]:
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
from rasterio.enums import Resampling

# Set matplotlib config directory to avoid permission issues
os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"

print("✓ Libraries imported successfully")
print(f"  rasterio version: {rasterio.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  rasterio version: 1.5.0
  NumPy version: 1.26.4


## Define CORINE Class Mapping

The CORINE raster uses simplified codes 1-44 (not the full CLC codes like 111, 112, etc.).

In [5]:
# CORINE class descriptions (simplified codes 1-44 used in raster)
CORINE_CLASSES = {
    # Artificial surfaces (1-11)
    1: "Continuous urban fabric",
    2: "Discontinuous urban fabric",
    3: "Industrial or commercial units",
    4: "Road and rail networks",
    5: "Port areas",
    6: "Airports",
    7: "Mineral extraction sites",
    8: "Dump sites",
    9: "Construction sites",
    10: "Green urban areas",
    11: "Sport and leisure facilities",
    
    # Agricultural areas (12-22)
    12: "Non-irrigated arable land",
    13: "Permanently irrigated land",
    14: "Rice fields",
    15: "Vineyards",
    16: "Fruit trees and berry plantations",
    17: "Olive groves",
    18: "Pastures",
    19: "Annual crops with permanent crops",
    20: "Complex cultivation patterns",
    21: "Agriculture with natural vegetation",
    22: "Agro-forestry areas",
    
    # Forest and semi-natural areas (23-34)
    23: "Broad-leaved forest",
    24: "Coniferous forest",
    25: "Mixed forest",
    26: "Natural grasslands",
    27: "Moors and heathland",
    28: "Sclerophyllous vegetation",
    29: "Transitional woodland-shrub",
    30: "Beaches, dunes, sands",
    31: "Bare rocks",
    32: "Sparsely vegetated areas",
    33: "Burnt areas",
    34: "Glaciers and perpetual snow",
    
    # Wetlands (35-39)
    35: "Inland marshes",
    36: "Peat bogs",
    37: "Salt marshes",
    38: "Salines",
    39: "Intertidal flats",
    
    # Water bodies (40-44)
    40: "Water courses",
    41: "Water bodies",
    42: "Coastal lagoons",
    43: "Estuaries",
    44: "Sea and ocean",
    
    48: "No data"
}

# Color mapping for visualization
CORINE_COLORS = {
    1: '#E6004D', 2: '#FF0000', 3: '#CC4DF2', 4: '#CC0000', 5: '#E6CCCC',
    6: '#E6CCE6', 7: '#A600CC', 8: '#A64DCC', 9: '#FF4DFF', 10: '#FFA6FF',
    11: '#FFE6FF', 12: '#FFFFA8', 13: '#FFFF00', 14: '#E6E600', 15: '#E68000',
    16: '#F2A64D', 17: '#E6A600', 18: '#E6E64D', 19: '#FFE6A6', 20: '#FFE64D',
    21: '#E6CC4D', 22: '#F2CCA6', 23: '#80FF00', 24: '#00A600', 25: '#4DFF00',
    26: '#CCF24D', 27: '#A6FF80', 28: '#A6E64D', 29: '#A6F200', 30: '#E6E6E6',
    31: '#CCCCCC', 32: '#CCFFCC', 33: '#000000', 34: '#A6E6CC', 35: '#A6A6FF',
    36: '#4D4DFF', 37: '#CCCCFF', 38: '#E6E6FF', 39: '#A6A6E6', 40: '#00CCF2',
    41: '#80F2E6', 42: '#00FFA6', 43: '#A6FFE6', 44: '#E6F2FF'
}

print(f"✓ Defined {len(CORINE_CLASSES)} CORINE land cover classes")

✓ Defined 45 CORINE land cover classes


## Configure Processing Parameters

Setting which Sentinel-2 tiles to process and the output directory for aligned data.

In [8]:
# --- Path Configuration ---
# Directory containing your downloaded Sentinel-2 .SAFE folders
S2_DATA_DIR = f"/p/scratch/training2600/team1/data/T33VWH"

# CORINE Land Cover raster (shared location)
CORINE_PATH = "/p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif"

# Output directory for processed data (stacked bands + aligned CORINE)
OUTPUT_DIR = f"/p/scratch/training2600/team1/data/T33VWH/aligned_data"

# --- Tile Selection ---
# Options:
#   'all'  - Process all .SAFE directories found in S2_DATA_DIR
#   'list' - Process only tiles listed in TILE_LIST below
TILE_SELECTION = 'all'

# If TILE_SELECTION = 'list', specify which tiles to process:
TILE_LIST = [
    # Add your specific tile names here, e.g.:
    # "S2A_MSIL2A_20181019T102031_N0500_R065_T33UUP_20230813T105225.SAFE",
]

# --- Processing Options ---
SKIP_EXISTING = True  # Skip tiles that already have output files

# ============================================================
print("Configuration:")
print(f"  S2 Data Directory: {S2_DATA_DIR}")
print(f"  CORINE Path: {CORINE_PATH}")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Tile Selection: {TILE_SELECTION}")

Configuration:
  S2 Data Directory: /p/scratch/training2600/team1/data/T33VWH
  CORINE Path: /p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif
  Output Directory: /p/scratch/training2600/team1/data/T33VWH/aligned_data
  Tile Selection: all


## Discover Available Tiles

In [9]:
def find_s2_tiles(data_dir, selection='all', tile_list=None):
    """
    Find Sentinel-2 .SAFE directories to process.
    
    Parameters:
    -----------
    data_dir : str
        Directory containing .SAFE folders
    selection : str
        'all' to find all tiles, 'list' to use tile_list
    tile_list : list
        List of specific tile names to process
    
    Returns:
    --------
    list : List of Path objects for .SAFE directories
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"❌ Data directory not found: {data_dir}")
        return []
    
    if selection == 'all':
        tiles = sorted(data_path.glob("*.SAFE"))
    else:
        tiles = []
        for tile_name in (tile_list or []):
            tile_path = data_path / tile_name
            if tile_path.exists():
                tiles.append(tile_path)
            else:
                print(f"⚠ Tile not found: {tile_name}")
    
    return tiles

# Find tiles
available_tiles = find_s2_tiles(S2_DATA_DIR, TILE_SELECTION, TILE_LIST)

print(f"\n📂 Found {len(available_tiles)} Sentinel-2 tile(s) to process:")
for i, tile in enumerate(available_tiles, 1):
    print(f"  {i}. {tile.name}")

if len(available_tiles) == 0:
    print("\n⚠ No tiles found! Check your S2_DATA_DIR path.")


📂 Found 4 Sentinel-2 tile(s) to process:
  1. S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE
  2. S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511.SAFE
  3. S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE
  4. S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146.SAFE


## Verify CORINE Data

In [10]:
# Check CORINE file exists and get basic info
corine_path = Path(CORINE_PATH)

if not corine_path.exists():
    print(f"❌ CORINE file not found: {CORINE_PATH}")
    print("\nPlease check the path or download CORINE data.")
else:
    with rasterio.open(corine_path) as src:
        print("✓ CORINE Land Cover 2018")
        print(f"  File: {corine_path.name}")
        print(f"  Size: {src.width} x {src.height} pixels")
        print(f"  Resolution: {src.res[0]}m x {src.res[1]}m")
        print(f"  CRS: {src.crs}")
        print(f"  NoData value: {src.nodata}")
        print(f"  Bounds: {src.bounds}")

✓ CORINE Land Cover 2018
  File: U2018_CLC2018_V2020_20u1.tif
  Size: 65000 x 46000 pixels
  Resolution: 100.0m x 100.0m
  CRS: PROJCS["ETRS89-extended / LAEA Europe",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",52],PARAMETER["longitude_of_center",10],PARAMETER["false_easting",4321000],PARAMETER["false_northing",3210000],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3035"]]
  NoData value: -128.0
  Bounds: BoundingBox(left=900000.0, bottom=900000.0, right=7400000.0, top=5500000.0)


## Stack Sentinel-2 Bands

Combine the 10m resolution bands (B02, B03, B04, B08) into a single multi-band GeoTIFF using rasterio.

In [11]:
def stack_s2_bands(safe_path, output_path):
    """
    Stack Sentinel-2 bands (B02, B03, B04, B08) into a single GeoTIFF.
    
    Parameters:
    -----------
    safe_path : Path
        Path to .SAFE directory
    output_path : Path
        Output path for stacked GeoTIFF
    
    Returns:
    --------
    dict : Info about the stacked file (crs, bounds, etc.) or None on failure
    """
    bands = ['B02', 'B03', 'B04', 'B08']  # Blue, Green, Red, NIR
    resolution = '10m'
    
    try:
        # Find band files
        band_paths = {}
        for band_name in bands:
            pattern = f"**/R{resolution}/*_{band_name}_{resolution}.jp2"
            matches = list(safe_path.glob(pattern))
            if matches:
                band_paths[band_name] = matches[0]
            else:
                print(f"  ⚠ {band_name} not found")
                return None
        
        # Read first band for metadata
        with rasterio.open(band_paths['B02']) as src:
            profile = src.profile.copy()
            height, width = src.height, src.width
            crs = src.crs
            transform = src.transform
            bounds = src.bounds
        
        # Update profile for output
        profile.update(
            driver='GTiff',
            count=len(bands),
            dtype='uint16',
            compress='lzw',
            tiled=True,
            blockxsize=256,
            blockysize=256
        )
        
        # Create output and write bands
        with rasterio.open(output_path, 'w', **profile) as dst:
            for i, (band_name, band_path) in enumerate(band_paths.items(), start=1):
                with rasterio.open(band_path) as src:
                    data = src.read(1)
                    dst.write(data, i)
                    dst.set_band_description(i, band_name)
        
        return {
            'crs': crs,
            'width': width,
            'height': height,
            'transform': transform,
            'bounds': bounds,
            'bands': bands
        }
        
    except Exception as e:
        print(f"  ❌ Band stacking failed: {e}")
        import traceback
        traceback.print_exc()
        return None


print("✓ Band stacking function defined")

✓ Band stacking function defined


## Align CORINE to Sentinel-2

Reproject CORINE from EPSG:3035 to match the S2 tile's UTM projection and extent using rasterio's warp functionality.

In [12]:
def align_corine_to_s2(corine_path, s2_path, output_path):
    """
    Align CORINE raster to match Sentinel-2 tile geometry using rasterio.
    
    Parameters:
    -----------
    corine_path : Path
        Path to CORINE GeoTIFF
    s2_path : Path
        Path to stacked S2 GeoTIFF (reference)
    output_path : Path
        Output path for aligned CORINE
    
    Returns:
    --------
    bool : Success status
    """
    try:
        # Get S2 geometry (target)
        with rasterio.open(s2_path) as s2_src:
            dst_crs = s2_src.crs
            dst_transform = s2_src.transform
            dst_width = s2_src.width
            dst_height = s2_src.height
            dst_bounds = s2_src.bounds
        
        print(f"    Target CRS: {dst_crs}")
        print(f"    Target size: {dst_width} x {dst_height} pixels")
        print(f"    Target bounds: {dst_bounds}")
        
        # Open CORINE and reproject
        with rasterio.open(corine_path) as src:
            # Create output profile
            out_profile = src.profile.copy()
            out_profile.update(
                driver='GTiff',
                crs=dst_crs,
                transform=dst_transform,
                width=dst_width,
                height=dst_height,
                compress='lzw',
                tiled=True
            )
            
            # Create output raster
            with rasterio.open(output_path, 'w', **out_profile) as dst:
                # Reproject using nearest neighbor (for categorical data)
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=dst_transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )
        
        return True
        
    except Exception as e:
        print(f"  ❌ CORINE alignment failed: {e}")
        import traceback
        traceback.print_exc()
        return False


def check_corine_coverage(corine_path):
    """
    Check if aligned CORINE has valid data.
    
    Returns:
    --------
    tuple : (has_valid_data, unique_classes, coverage_percent)
    """
    with rasterio.open(corine_path) as src:
        data = src.read(1)
    
    total_pixels = data.size
    valid_mask = (data >= 1) & (data <= 44)
    valid_pixels = np.sum(valid_mask)
    coverage_percent = (valid_pixels / total_pixels) * 100
    
    unique = np.unique(data[valid_mask])
    
    return len(unique) > 0, unique, coverage_percent


print("✓ CORINE alignment functions defined")

✓ CORINE alignment functions defined


## Process Tiles

### Main Processing Loop

This cell will process all selected tiles: 

1. Stack S2 bands into multi-band GeoTIFF
2. Reproject and align CORINE to match S2 geometry
3. Verify alignment and report coverage

In [13]:
def process_tile_alignment(safe_path, corine_path, output_dir, skip_existing=True):
    """
    Process a single Sentinel-2 tile: stack bands and align CORINE.
    
    Returns:
    --------
    dict : Processing results
    """
    tile_name = safe_path.stem
    result = {
        'tile': tile_name,
        'status': 'unknown',
        's2_stacked': None,
        'corine_aligned': None,
        'crs': None,
        'n_classes': 0,
        'coverage_percent': 0
    }
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Output paths
    s2_stacked = output_dir / f"{tile_name}_stacked.tif"
    corine_aligned = output_dir / f"corine_aligned_{tile_name}.tif"
    
    # Check for existing output
    if skip_existing and s2_stacked.exists() and corine_aligned.exists():
        print(f"  ⏭ Skipping (outputs exist)")
        has_coverage, classes, coverage = check_corine_coverage(corine_aligned)
        result['status'] = 'skipped'
        result['s2_stacked'] = str(s2_stacked)
        result['corine_aligned'] = str(corine_aligned)
        result['n_classes'] = len(classes)
        result['coverage_percent'] = coverage
        return result
    
    # Step 1: Stack bands
    print(f"  📦 Stacking S2 bands...")
    stack_info = stack_s2_bands(safe_path, s2_stacked)
    if stack_info is None:
        result['status'] = 'failed_stacking'
        return result
    
    result['crs'] = str(stack_info['crs'])
    print(f"    Created: {s2_stacked.name}")
    print(f"    Size: {stack_info['width']} x {stack_info['height']} pixels")
    print(f"    CRS: {stack_info['crs']}")
    
    # Step 2: Align CORINE
    print(f"  🗺️ Aligning CORINE to S2 geometry...")
    if not align_corine_to_s2(corine_path, s2_stacked, corine_aligned):
        result['status'] = 'failed_alignment'
        return result
    
    # Step 3: Check coverage
    print(f"  🔍 Checking CORINE coverage...")
    has_coverage, classes, coverage = check_corine_coverage(corine_aligned)
    
    if not has_coverage:
        print(f"  ⚠ No valid CORINE data (tile outside coverage?)")
        result['status'] = 'no_coverage'
        return result
    
    result['status'] = 'success'
    result['s2_stacked'] = str(s2_stacked)
    result['corine_aligned'] = str(corine_aligned)
    result['n_classes'] = len(classes)
    result['coverage_percent'] = coverage
    
    print(f"    CORINE classes found: {len(classes)}")
    print(f"    Valid coverage: {coverage:.1f}%")
    
    return result


print("✓ Tile processing function defined")

✓ Tile processing function defined


In [14]:
# ============================================================
# MAIN PROCESSING LOOP
# ============================================================

print("="*70)
print("ALIGNING CORINE WITH SENTINEL-2 TILES")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tiles to process: {len(available_tiles)}")
print(f"Output directory: {OUTPUT_DIR}")
print()

# Store results
all_results = []

# Process each tile
for i, tile_path in enumerate(available_tiles, 1):
    print(f"\n[{i}/{len(available_tiles)}] Processing: {tile_path.name}")
    
    result = process_tile_alignment(
        safe_path=tile_path,
        corine_path=CORINE_PATH,
        output_dir=OUTPUT_DIR,
        skip_existing=SKIP_EXISTING
    )
    
    all_results.append(result)
    
    if result['status'] == 'success':
        print(f"  ✅ Success: {result['n_classes']} classes, {result['coverage_percent']:.1f}% coverage")
    elif result['status'] == 'skipped':
        print(f"  ⏭ Skipped (already processed)")
    else:
        print(f"  ❌ Status: {result['status']}")

# Summary
print("\n" + "="*70)
print("ALIGNMENT SUMMARY")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

successful = [r for r in all_results if r['status'] == 'success']
skipped = [r for r in all_results if r['status'] == 'skipped']
failed = [r for r in all_results if r['status'] not in ['success', 'skipped']]

print(f"\nResults:")
print(f"  ✅ Successful: {len(successful)}")
print(f"  ⏭ Skipped: {len(skipped)}")
print(f"  ❌ Failed: {len(failed)}")
print(f"\n  Output directory: {OUTPUT_DIR}")

ALIGNING CORINE WITH SENTINEL-2 TILES
Start time: 2026-03-10 11:31:57
Tiles to process: 4
Output directory: /p/scratch/training2600/team1/data/T33VWH/aligned_data


[1/4] Processing: S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE
  📦 Stacking S2 bands...
    Created: S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912_stacked.tif
    Size: 10980 x 10980 pixels
    CRS: EPSG:32633
  🗺️ Aligning CORINE to S2 geometry...
    Target CRS: EPSG:32633
    Target size: 10980 x 10980 pixels
    Target bounds: BoundingBox(left=499980.0, bottom=6690240.0, right=609780.0, top=6800040.0)
  🔍 Checking CORINE coverage...
    CORINE classes found: 22
    Valid coverage: 100.0%
  ✅ Success: 22 classes, 100.0% coverage

[2/4] Processing: S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511.SAFE
  📦 Stacking S2 bands...
    Created: S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511_stacked.tif
    Size: 10980 x 10980 pixels
    CRS: EPSG:32633
  🗺️ Aligning 

## Visualize Alignment Results

In [15]:
def visualize_alignment(s2_path, corine_path, title="Alignment Verification", save_path=None):
    """
    Visualize the S2 image and aligned CORINE side by side.
    
    Parameters:
    -----------
    s2_path : str or Path
        Path to stacked S2 GeoTIFF
    corine_path : str or Path
        Path to aligned CORINE GeoTIFF
    title : str
        Plot title
    save_path : str or Path, optional
        Path to save the figure (PNG format)
    """
    # Load S2 data
    with rasterio.open(s2_path) as src:
        # Read RGB bands (B04, B03, B02 = bands 3, 2, 1)
        r = src.read(3).astype(float)  # B04 (Red)
        g = src.read(2).astype(float)  # B03 (Green)
        b = src.read(1).astype(float)  # B02 (Blue)
    
    # Create RGB composite
    rgb = np.stack([r, g, b], axis=-1)
    p2, p98 = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
    
    # Load CORINE data
    with rasterio.open(corine_path) as src:
        corine = src.read(1)
    
    # Create CORINE colormap
    from matplotlib.colors import ListedColormap
    unique_classes = np.unique(corine[(corine >= 1) & (corine <= 44)])
    colors = [CORINE_COLORS.get(c, '#888888') for c in range(45)]
    cmap = ListedColormap(colors)
    
    # Create figure (subsample for display)
    step = 10
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # S2 RGB
    axes[0].imshow(rgb[::step, ::step])
    axes[0].set_title('Sentinel-2 RGB (B04, B03, B02)', fontsize=12)
    axes[0].axis('off')
    
    # CORINE
    axes[1].imshow(corine[::step, ::step], cmap=cmap, vmin=0, vmax=44)
    axes[1].set_title('Aligned CORINE Land Cover', fontsize=12)
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(rgb[::step, ::step])
    corine_masked = np.ma.masked_where((corine < 1) | (corine > 44), corine)
    axes[2].imshow(corine_masked[::step, ::step], cmap=cmap, vmin=0, vmax=44, alpha=0.4)
    axes[2].set_title('S2 + CORINE Overlay', fontsize=12)
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"✓ Figure saved to: {save_path}")
    
    plt.show()
    
    # Print class distribution
    print("\nCORINE Classes in this tile:")
    for c in sorted(unique_classes):
        count = np.sum(corine == c)
        pct = count / corine.size * 100
        print(f"  Class {c:2d}: {CORINE_CLASSES.get(c, 'Unknown')[:30]:30s} - {count:,} pixels ({pct:.1f}%)")


# Results directory for saving figures
RESULTS_DIR = f"/p/project1/training2600/team1/results"

Visualizing: S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230...
✓ Figure saved to: /p/project1/training2600/team1/results/alignment_S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230.png

CORINE Classes in this tile:
  Class  2: Discontinuous urban fabric     - 1,783,363 pixels (1.5%)
  Class  3: Industrial or commercial units - 221,051 pixels (0.2%)
  Class  4: Road and rail networks         - 6,491 pixels (0.0%)
  Class  6: Airports                       - 34,575 pixels (0.0%)
  Class  7: Mineral extraction sites       - 54,661 pixels (0.0%)
  Class  8: Dump sites                     - 19,387 pixels (0.0%)
  Class  9: Construction sites             - 3,295 pixels (0.0%)
  Class 10: Green urban areas              - 49,814 pixels (0.0%)
  Class 11: Sport and leisure facilities   - 127,916 pixels (0.1%)
  Class 12: Non-irrigated arable land      - 7,197,020 pixels (6.0%)
  Class 16: Fruit trees and berry plantati - 3,002 pixels (0.0%)
  Class 18: Pastures                       - 180,

In [19]:
# Visualize first (or other) successful result
successful_results = [r for r in all_results if r['status'] in ['success', 'skipped']]

if successful_results:
    result = successful_results[0] # Change to change which result is visualized
    tile_name = result['tile']
    
    # Create save path for the visualization
    save_path = Path(RESULTS_DIR) / f"alignment_{tile_name[:50]}.png"
    
    print(f"Visualizing: {tile_name[:50]}...")
    visualize_alignment(
        result['s2_stacked'], 
        result['corine_aligned'],
        title=f"Alignment: {tile_name[:40]}...",
        save_path=save_path
    )
else:
    print("No successful results to visualize.")

Visualizing: S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230...
✓ Figure saved to: /p/project1/training2600/team1/results/alignment_S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230.png

CORINE Classes in this tile:
  Class  2: Discontinuous urban fabric     - 1,783,363 pixels (1.5%)
  Class  3: Industrial or commercial units - 221,051 pixels (0.2%)
  Class  4: Road and rail networks         - 6,491 pixels (0.0%)
  Class  6: Airports                       - 34,575 pixels (0.0%)
  Class  7: Mineral extraction sites       - 54,661 pixels (0.0%)
  Class  8: Dump sites                     - 19,387 pixels (0.0%)
  Class  9: Construction sites             - 3,295 pixels (0.0%)
  Class 10: Green urban areas              - 49,814 pixels (0.0%)
  Class 11: Sport and leisure facilities   - 127,916 pixels (0.1%)
  Class 12: Non-irrigated arable land      - 7,197,020 pixels (6.0%)
  Class 16: Fruit trees and berry plantati - 3,002 pixels (0.0%)
  Class 18: Pastures                       - 180,

## View Processing Results Summary

In [16]:
# Display detailed results
print("Detailed Results per Tile:")
print("-" * 90)
print(f"{'Tile':<50} {'Status':<15} {'Classes':<10} {'Coverage':<10}")
print("-" * 90)

for result in all_results:
    tile_short = result['tile'][:48] + ".." if len(result['tile']) > 50 else result['tile']
    coverage = f"{result['coverage_percent']:.1f}%" if result['coverage_percent'] > 0 else "-"
    classes = str(result['n_classes']) if result['n_classes'] > 0 else "-"
    print(f"{tile_short:<50} {result['status']:<15} {classes:<10} {coverage:<10}")

print("-" * 90)

# List output files
print("\n📁 Output Files:")
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    for f in sorted(output_path.glob("*.tif")):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.1f} MB)")
else:
    print(f"  Output directory not found: {OUTPUT_DIR}")

Detailed Results per Tile:
------------------------------------------------------------------------------------------
Tile                                               Status          Classes    Coverage  
------------------------------------------------------------------------------------------
S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_202.. success         22         100.0%    
S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_202.. success         22         100.0%    
S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_202.. success         22         100.0%    
S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_202.. success         22         100.0%    
------------------------------------------------------------------------------------------

📁 Output Files:
  S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912_stacked.tif (2743.5 MB)
  S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511_stacked.tif (2164.1 MB)
  S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056_s

# Checkpoint 2: Step 2

## Configure Parameters

Configuring
1. Path to aligned data from Lab 4.1
2. Patch extraction parameters
3. Normalization settings
4. Output directory for training data

In [43]:
# --- Path Configuration ---
# Directory containing aligned data from Lab 4.1
ALIGNED_DATA_DIR = f"/p/scratch/training2600/team1/data/T33VWH/aligned_data"

# Output directory for training patches
OUTPUT_DIR = f"/p/scratch/training2600/team1/data/T33VWH/training_data"

# --- Tile Selection ---
# Options:
#   'all'  - Process all aligned tile pairs found
#   'list' - Process only tiles listed in TILE_LIST below
TILE_SELECTION = 'all'

# If TILE_SELECTION = 'list', specify which tiles to process:
TILE_LIST = [
    # Add your specific tile names here (without _stacked.tif suffix)
]

# --- Patch Extraction Parameters ---
PATCH_SIZE = 3       # Size in pixels (3 = 30m x 30m at 10m resolution)
STRIDE = None        # Stride for extraction (None = same as PATCH_SIZE, no overlap)
MAX_PATCHES = 100000  # Maximum patches per tile (increased to improve training quality)

# --- Normalization Settings ---
# Options: 'minmax', 'zscore', 'percentile', 'none'
NORMALIZATION = 'percentile'

# For percentile normalization: clip to these percentiles
PERCENTILE_LOW = 2
PERCENTILE_HIGH = 98

# --- Processing Options ---
SKIP_EXISTING = True  # Skip tiles that already have output files

# ============================================================
print("Configuration:")
print(f"  Aligned Data Directory: {ALIGNED_DATA_DIR}")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Tile Selection: {TILE_SELECTION}")
print(f"  Patch Size: {PATCH_SIZE}x{PATCH_SIZE} ({PATCH_SIZE * 10}m x {PATCH_SIZE * 10}m)")
print(f"  Max Patches per Tile: {MAX_PATCHES:,}")
print(f"  Normalization: {NORMALIZATION}")

Configuration:
  Aligned Data Directory: /p/scratch/training2600/team1/data/T33VWH/aligned_data
  Output Directory: /p/scratch/training2600/team1/data/T33VWH/training_data
  Tile Selection: all
  Patch Size: 3x3 (30m x 30m)
  Max Patches per Tile: 100,000
  Normalization: percentile


## Discover Available Aligned Data

In [44]:
def find_aligned_pairs(data_dir, selection='all', tile_list=None):
    """
    Find pairs of stacked S2 and aligned CORINE files.
    
    Returns:
    --------
    list : List of dicts with 's2_path', 'corine_path', 'tile_name'
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"❌ Data directory not found: {data_dir}")
        print("\n💡 Have you completed Lab 4.1 to create aligned data?")
        return []
    
    # Find all stacked S2 files
    s2_files = sorted(data_path.glob("*_stacked.tif"))
    
    pairs = []
    for s2_path in s2_files:
        # Extract tile name
        tile_name = s2_path.stem.replace('_stacked', '')
        
        # Check if we should include this tile
        if selection == 'list' and tile_list:
            if tile_name not in tile_list:
                continue
        
        # Find matching CORINE file
        corine_path = data_path / f"corine_aligned_{tile_name}.tif"
        
        if corine_path.exists():
            pairs.append({
                's2_path': s2_path,
                'corine_path': corine_path,
                'tile_name': tile_name
            })
        else:
            print(f"⚠ Missing CORINE for: {tile_name}")
    
    return pairs

# Find aligned pairs
aligned_pairs = find_aligned_pairs(ALIGNED_DATA_DIR, TILE_SELECTION, TILE_LIST)

print(f"\n📂 Found {len(aligned_pairs)} aligned tile pair(s):")
for i, pair in enumerate(aligned_pairs, 1):
    print(f"  {i}. {pair['tile_name'][:60]}...")

if len(aligned_pairs) == 0:
    print("\n⚠ No aligned data found!")


📂 Found 4 aligned tile pair(s):
  1. S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912...
  2. S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511...
  3. S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056...
  4. S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230709T051146...


## Defining normalization functions

In [45]:
def normalize_minmax(data, min_val=0, max_val=10000):
    """
    Min-max normalization to [0, 1] range.
    
    Parameters:
    -----------
    data : np.ndarray
        Input data (any shape)
    min_val : float
        Minimum value for scaling (default 0 for S2)
    max_val : float
        Maximum value for scaling (default 10000 for S2 L2A)
    
    Returns:
    --------
    np.ndarray : Normalized data in [0, 1]
    """
    data = data.astype(np.float32)
    normalized = (data - min_val) / (max_val - min_val + 1e-6)
    return np.clip(normalized, 0, 1)


def normalize_zscore(data, mean=None, std=None):
    """
    Z-score normalization (standardization).
    
    Parameters:
    -----------
    data : np.ndarray
        Input data
    mean : float or None
        Mean value (computed if None)
    std : float or None
        Std value (computed if None)
    
    Returns:
    --------
    tuple : (normalized_data, mean, std)
    """
    data = data.astype(np.float32)
    if mean is None:
        mean = np.mean(data)
    if std is None:
        std = np.std(data)
    normalized = (data - mean) / (std + 1e-6)
    return normalized, mean, std


def normalize_percentile(data, low_pct=2, high_pct=98):
    """
    Percentile-based normalization (robust to outliers).
    
    Parameters:
    -----------
    data : np.ndarray
        Input data
    low_pct : float
        Low percentile for clipping
    high_pct : float
        High percentile for clipping
    
    Returns:
    --------
    tuple : (normalized_data, low_val, high_val)
    """
    data = data.astype(np.float32)
    low_val = np.percentile(data, low_pct)
    high_val = np.percentile(data, high_pct)
    normalized = (data - low_val) / (high_val - low_val + 1e-6)
    normalized = np.clip(normalized, 0, 1)
    return normalized, low_val, high_val


def normalize_data(data, method='percentile', **kwargs):
    """
    Normalize data using specified method.
    
    Parameters:
    -----------
    data : np.ndarray
        Input data
    method : str
        'minmax', 'zscore', 'percentile', or 'none'
    **kwargs : dict
        Additional arguments for the normalization method
    
    Returns:
    --------
    tuple : (normalized_data, norm_params)
    """
    if method == 'minmax':
        normalized = normalize_minmax(data, **kwargs)
        params = {'method': 'minmax', **kwargs}
    elif method == 'zscore':
        normalized, mean, std = normalize_zscore(data, **kwargs)
        params = {'method': 'zscore', 'mean': float(mean), 'std': float(std)}
    elif method == 'percentile':
        low_pct = kwargs.get('low_pct', 2)
        high_pct = kwargs.get('high_pct', 98)
        normalized, low_val, high_val = normalize_percentile(data, low_pct, high_pct)
        params = {'method': 'percentile', 'low_pct': low_pct, 'high_pct': high_pct,
                  'low_val': float(low_val), 'high_val': float(high_val)}
    else:  # 'none'
        normalized = data.astype(np.float32)
        params = {'method': 'none'}
    
    return normalized, params


print("✓ Normalization functions defined")
print("\nAvailable methods:")
print("  - minmax: Scale to [0, 1] using fixed min/max")
print("  - zscore: Standardize to mean=0, std=1")
print("  - percentile: Robust scaling using percentiles")
print("  - none: Keep original values (as float32)")

✓ Normalization functions defined

Available methods:
  - minmax: Scale to [0, 1] using fixed min/max
  - zscore: Standardize to mean=0, std=1
  - percentile: Robust scaling using percentiles
  - none: Keep original values (as float32)


## Define Extraction Function

In [46]:
def extract_patches(s2_path, corine_path, patch_size=3, stride=None, max_patches=50000,
                    normalization='percentile', norm_kwargs=None):
    """
    Extract training patches from aligned S2 imagery with CORINE labels.
    
    Parameters:
    -----------
    s2_path : Path
        Path to stacked S2 GeoTIFF
    corine_path : Path
        Path to aligned CORINE GeoTIFF
    patch_size : int
        Patch size in pixels
    stride : int
        Stride for extraction (None = patch_size)
    max_patches : int
        Maximum patches to extract
    normalization : str
        Normalization method
    norm_kwargs : dict
        Additional arguments for normalization
    
    Returns:
    --------
    tuple : (patches, labels, metadata) or (None, None, None)
    """
    if norm_kwargs is None:
        norm_kwargs = {}
    
    try:
        # Open datasets using rasterio
        with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
            n_bands = s2_src.count
            height = s2_src.height
            width = s2_src.width
            
            if stride is None:
                stride = patch_size
            
            print(f"    Image size: {width} x {height} pixels")
            print(f"    Bands: {n_bands}")
            print(f"    Patch size: {patch_size}, Stride: {stride}")
            
            # Read all data into memory (for efficiency)
            # Shape: (bands, height, width)
            s2_data = s2_src.read()
            corine_data = corine_src.read(1)
        
        # Apply normalization to entire image first (for consistent statistics)
        print(f"    Applying {normalization} normalization...")
        s2_normalized, norm_params = normalize_data(s2_data, normalization, **norm_kwargs)
        
        # Extract patches
        patches, labels = [], []
        
        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                # Extract patch (bands, h, w) -> (h, w, bands)
                patch = s2_normalized[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))  # (H, W, C)
                
                # Get center pixel label
                center_y = y + patch_size // 2
                center_x = x + patch_size // 2
                label = corine_data[center_y, center_x]
                
                # Skip invalid labels (must be 1-44)
                if label < 1 or label > 44:
                    continue
                
                # Skip patches with missing S2 data (check original before normalization)
                original_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                if np.any(original_patch == 0):
                    continue
                
                patches.append(patch)
                labels.append(label)
                
                if len(patches) >= max_patches:
                    break
            
            if len(patches) >= max_patches:
                break
            
            # Progress update
            if y % 2000 == 0 and y > 0:
                print(f"    Progress: {len(patches):,} patches extracted...")
        
        if len(patches) == 0:
            return None, None, None
        
        # Convert to arrays
        patches = np.array(patches, dtype=np.float32)
        labels = np.array(labels, dtype=np.uint8)
        
        # Create metadata
        unique_labels, counts = np.unique(labels, return_counts=True)
        metadata = {
            's2_file': Path(s2_path).name,
            'corine_file': Path(corine_path).name,
            'patch_size': patch_size,
            'stride': stride,
            'n_patches': len(patches),
            'n_bands': patches.shape[3],
            'n_classes': len(unique_labels),
            'label_distribution': {int(k): int(v) for k, v in zip(unique_labels, counts)},
            'extraction_date': datetime.now().isoformat(),
            'patch_shape': list(patches.shape),
            'bands': ['B02', 'B03', 'B04', 'B08'],
            'normalization': norm_params
        }
        
        return patches, labels, metadata
        
    except Exception as e:
        print(f"  ❌ Patch extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None


print("✓ Patch extraction function defined")

✓ Patch extraction function defined


## Process Tile Function

In [47]:
def process_tile_extraction(pair, output_dir, patch_size, stride, max_patches,
                            normalization, norm_kwargs, skip_existing=True):
    """
    Extract patches from a single aligned tile pair.
    
    Returns:
    --------
    dict : Processing results
    """
    tile_name = pair['tile_name']
    s2_path = pair['s2_path']
    corine_path = pair['corine_path']
    
    result = {
        'tile': tile_name,
        'status': 'unknown',
        'n_patches': 0,
        'n_classes': 0,
        'output_file': None,
        'metadata': None
    }
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Output paths
    output_npz = output_dir / f"patches_{tile_name}_data.npz"
    metadata_file = output_dir / f"patches_{tile_name}_metadata.json"
    
    # Check for existing output
    if skip_existing and output_npz.exists():
        print(f"  ⏭ Skipping (output exists)")
        if metadata_file.exists():
            with open(metadata_file) as f:
                result['metadata'] = json.load(f)
                result['n_patches'] = result['metadata'].get('n_patches', 0)
                result['n_classes'] = result['metadata'].get('n_classes', 0)
        result['status'] = 'skipped'
        result['output_file'] = str(output_npz)
        return result
    
    # Extract patches
    print(f"  ✂️ Extracting patches...")
    patches, labels, metadata = extract_patches(
        s2_path, corine_path, 
        patch_size=patch_size, 
        stride=stride, 
        max_patches=max_patches,
        normalization=normalization,
        norm_kwargs=norm_kwargs
    )
    
    if patches is None:
        result['status'] = 'no_patches'
        return result
    
    # Save results
    print(f"  💾 Saving {len(patches):,} patches...")
    np.savez_compressed(output_npz, patches=patches, labels=labels)
    
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    result['status'] = 'success'
    result['n_patches'] = len(patches)
    result['n_classes'] = metadata['n_classes']
    result['output_file'] = str(output_npz)
    result['metadata'] = metadata
    
    return result


print("✓ Tile processing function defined")

✓ Tile processing function defined


## Main Processing Loop

In [48]:
# ============================================================
# MAIN PROCESSING LOOP
# ============================================================

print("="*70)
print("EXTRACTING TRAINING PATCHES")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tiles to process: {len(aligned_pairs)}")
print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE} pixels")
print(f"Max patches per tile: {MAX_PATCHES:,}")
print(f"Normalization: {NORMALIZATION}")
print()

# Prepare normalization kwargs (using percentile method)
norm_kwargs = {}
if NORMALIZATION == 'percentile':
    norm_kwargs = {'low_pct': PERCENTILE_LOW, 'high_pct': PERCENTILE_HIGH}

# Store results
all_results = []

# Process each tile
for i, pair in enumerate(aligned_pairs, 1):
    print(f"\n[{i}/{len(aligned_pairs)}] Processing: {pair['tile_name'][:50]}...")
    
    result = process_tile_extraction(
        pair=pair,
        output_dir=OUTPUT_DIR,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        max_patches=MAX_PATCHES,
        normalization=NORMALIZATION,
        norm_kwargs=norm_kwargs,
        skip_existing=SKIP_EXISTING
    )
    
    all_results.append(result)
    
    if result['status'] == 'success':
        print(f"  ✅ Success: {result['n_patches']:,} patches, {result['n_classes']} classes")
    elif result['status'] == 'skipped':
        print(f"  ⏭ Skipped (already processed): {result['n_patches']:,} patches")
    else:
        print(f"  ❌ Status: {result['status']}")

# Summary
print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

successful = [r for r in all_results if r['status'] == 'success']
skipped = [r for r in all_results if r['status'] == 'skipped']
failed = [r for r in all_results if r['status'] not in ['success', 'skipped']]

print(f"\nResults:")
print(f"  ✅ Successful: {len(successful)}")
print(f"  ⏭ Skipped: {len(skipped)}")
print(f"  ❌ Failed: {len(failed)}")

total_patches = sum(r['n_patches'] for r in all_results if r['n_patches'] > 0)
print(f"\n  Total patches extracted: {total_patches:,}")
print(f"  Output directory: {OUTPUT_DIR}")

EXTRACTING TRAINING PATCHES
Start time: 2026-03-10 12:24:11
Tiles to process: 4
Patch size: 3x3 pixels
Max patches per tile: 100,000
Normalization: percentile


[1/4] Processing: S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230...
  ✂️ Extracting patches...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 3, Stride: 3
    Applying percentile normalization...
  💾 Saving 100,000 patches...
  ✅ Success: 100,000 patches, 8 classes

[2/4] Processing: S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230...
  ✂️ Extracting patches...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 3, Stride: 3
    Applying percentile normalization...
  💾 Saving 100,000 patches...
  ✅ Success: 100,000 patches, 8 classes

[3/4] Processing: S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230...
  ✂️ Extracting patches...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 3, Stride: 3
    Applying percentile normalization...
  💾 Saving 100,000 patches...
  ✅ Succes

## View Processing Results

In [49]:
# Display detailed results
print("Detailed Results per Tile:")
print("-" * 80)

for result in all_results:
    status_icon = "✅" if result['status'] == 'success' else "⏭" if result['status'] == 'skipped' else "❌"
    print(f"{status_icon} {result['tile'][:50]}...")
    print(f"    Status: {result['status']}")
    if result['n_patches'] > 0:
        print(f"    Patches: {result['n_patches']:,}")
        print(f"    Classes: {result['n_classes']}")
        if result['output_file']:
            size_mb = Path(result['output_file']).stat().st_size / (1024 * 1024)
            print(f"    File: {Path(result['output_file']).name} ({size_mb:.1f} MB)")
        if result['metadata'] and 'normalization' in result['metadata']:
            norm = result['metadata']['normalization']
            print(f"    Normalization: {norm['method']}")
    print()

Detailed Results per Tile:
--------------------------------------------------------------------------------
✅ S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230...
    Status: success
    Patches: 100,000
    Classes: 8
    File: patches_S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912_data.npz (8.0 MB)
    Normalization: percentile

✅ S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230...
    Status: success
    Patches: 100,000
    Classes: 8
    File: patches_S2A_MSIL2A_20180604T103021_N0500_R108_T33VWH_20230723T023511_data.npz (6.5 MB)
    Normalization: percentile

✅ S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230...
    Status: success
    Patches: 100,000
    Classes: 8
    File: patches_S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056_data.npz (8.8 MB)
    Normalization: percentile

✅ S2B_MSIL2A_20180808T103019_N0500_R108_T33VWH_20230...
    Status: success
    Patches: 100,000
    Classes: 8
    File: patches_S2B_MSIL2A_20180808T103019_N0500_R108_T33V

## Visualize Sample Patches

In [50]:
def visualize_patches(patches, labels, n_samples=12, title="Sample Patches"):
    """
    Visualize sample patches with their labels.
    """
    # Get diverse samples (one or two per class)
    unique_labels = np.unique(labels)
    
    sample_indices = []
    for label in unique_labels:
        indices = np.where(labels == label)[0]
        n_from_class = min(2, len(indices))
        sample_indices.extend(np.random.choice(indices, n_from_class, replace=False))
        if len(sample_indices) >= n_samples:
            break
    
    sample_indices = sample_indices[:n_samples]
    
    # Create figure
    n_cols = 4
    n_rows = (len(sample_indices) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes
    
    for i, idx in enumerate(sample_indices):
        patch = patches[idx]
        label = labels[idx]
        
        # Create RGB composite (already normalized, just use R, G, B)
        # Bands are B02, B03, B04, B08 -> RGB is B04, B03, B02 = indices 2, 1, 0
        rgb = patch[:, :, [2, 1, 0]]
        rgb = np.clip(rgb, 0, 1)  # Ensure in [0, 1]
        
        # Upscale for visibility
        rgb_upscaled = np.repeat(np.repeat(rgb, 20, axis=0), 20, axis=1)
        
        axes[i].imshow(rgb_upscaled)
        class_name = CORINE_CLASSES.get(label, "Unknown")[:25]
        axes[i].set_title(f"Class {label}: {class_name}", fontsize=10)
        axes[i].axis('off')
    
    # Hide unused axes
    for i in range(len(sample_indices), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()

    output_dir = Path(os.getenv('PROJECT_training2600')) / 'team1/results'
    output_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_dir / "patches_viz.png")
    print("figure saved!")

In [51]:
# Load and visualize first (or other) successful result
successful_results = [r for r in all_results if r['status'] in ['success', 'skipped'] and r['output_file']]

if successful_results:
    result = successful_results[0] # Change to visualize other results
    data = np.load(result['output_file'])
    patches = data['patches']
    labels = data['labels']
    
    print(f"Loaded {len(patches):,} patches from: {Path(result['output_file']).name}")
    print(f"Patch value range: [{patches.min():.3f}, {patches.max():.3f}]")
    visualize_patches(patches, labels, n_samples=12, title=f"Sample Patches from {result['tile'][:40]}...")
else:
    print("No successful results to visualize.")

Loaded 100,000 patches from: patches_S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912_data.npz
Patch value range: [0.000, 1.000]
figure saved!


## Class Distribution Analysis

In [52]:
def plot_class_distribution(labels, title="Class Distribution"):
    """
    Plot the distribution of CORINE classes.
    """
    unique, counts = np.unique(labels, return_counts=True)
    
    # Sort by count
    sort_idx = np.argsort(counts)[::-1]
    unique = unique[sort_idx]
    counts = counts[sort_idx]
    
    # Get class names and colors
    class_names = [f"{c}: {CORINE_CLASSES.get(c, 'Unknown')[:20]}" for c in unique]
    colors = [CORINE_COLORS.get(c, '#888888') for c in unique]
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    bars = ax1.barh(range(len(unique)), counts, color=colors)
    ax1.set_yticks(range(len(unique)))
    ax1.set_yticklabels(class_names, fontsize=9)
    ax1.set_xlabel('Number of Patches')
    ax1.set_title('Patch Count by Class')
    ax1.invert_yaxis()
    
    # Add count labels
    for i, (count, bar) in enumerate(zip(counts, bars)):
        ax1.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=8)
    
    # Pie chart (top 10)
    top_n = min(10, len(unique))
    other_count = counts[top_n:].sum() if len(counts) > top_n else 0
    
    pie_counts = list(counts[:top_n])
    pie_labels = [f"{c}" for c in unique[:top_n]]
    pie_colors = colors[:top_n]
    
    if other_count > 0:
        pie_counts.append(other_count)
        pie_labels.append('Other')
        pie_colors.append('#888888')
    
    ax2.pie(pie_counts, labels=pie_labels, colors=pie_colors, 
            autopct='%1.1f%%', startangle=90)
    ax2.set_title('Top 10 Classes Distribution')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nStatistics:")
    print(f"  Total patches: {len(labels):,}")
    print(f"  Unique classes: {len(unique)}")
    print(f"  Most common: Class {unique[0]} ({CORINE_CLASSES.get(unique[0], 'Unknown')}) - {counts[0]:,} ({counts[0]/len(labels)*100:.1f}%)")
    print(f"  Least common: Class {unique[-1]} ({CORINE_CLASSES.get(unique[-1], 'Unknown')}) - {counts[-1]:,} ({counts[-1]/len(labels)*100:.1f}%)")
    
    # Class imbalance ratio
    imbalance_ratio = counts[0] / counts[-1]
    print(f"  Class imbalance ratio: {imbalance_ratio:.1f}:1")


# Analyze first successful result
if successful_results:
    result = successful_results[0]
    data = np.load(result['output_file'])
    labels = data['labels']
    
    plot_class_distribution(labels, title=f"Class Distribution: {result['tile'][:40]}...")
else:
    print("No successful results to analyze.")


Statistics:
  Total patches: 100,000
  Unique classes: 8
  Most common: Class 24 (Coniferous forest) - 67,535 (67.5%)
  Least common: Class 23 (Broad-leaved forest) - 695 (0.7%)
  Class imbalance ratio: 97.2:1


## Combine Multiple Tiles

In [53]:
def combine_datasets(result_list, output_path):
    """
    Combine patches from multiple tiles into a single dataset.
    """
    all_patches = []
    all_labels = []
    
    for result in result_list:
        if result['status'] not in ['success', 'skipped']:
            continue
        if not result['output_file'] or not Path(result['output_file']).exists():
            continue
        
        data = np.load(result['output_file'])
        patches = data['patches']
        labels = data['labels']
        
        all_patches.append(patches)
        all_labels.append(labels)
        
        print(f"  Added {len(labels):,} patches from {result['tile'][:40]}...")
    
    if not all_patches:
        print("No data to combine!")
        return None, None
    
    # Concatenate
    combined_patches = np.concatenate(all_patches, axis=0)
    combined_labels = np.concatenate(all_labels, axis=0)
    
    # Shuffle
    indices = np.random.permutation(len(combined_labels))
    combined_patches = combined_patches[indices]
    combined_labels = combined_labels[indices]
    
    # Save
    np.savez_compressed(output_path, patches=combined_patches, labels=combined_labels)
    
    print(f"\n✓ Combined dataset saved: {output_path}")
    print(f"  Total patches: {len(combined_labels):,}")
    print(f"  Unique classes: {len(np.unique(combined_labels))}")
    print(f"  File size: {Path(output_path).stat().st_size / (1024**2):.1f} MB")
    
    return combined_patches, combined_labels


# Combine all successful results
if len(successful_results) > 1:
    print("Combining datasets from multiple tiles...\n")
    combined_path = Path(OUTPUT_DIR) / "combined_training_data.npz"
    combined_patches, combined_labels = combine_datasets(all_results, combined_path)
    
    if combined_patches is not None:
        plot_class_distribution(combined_labels, title="Combined Dataset Class Distribution")
elif len(successful_results) == 1:
    print("Only one tile processed - no need to combine.")
    print(f"Your training data is at: {successful_results[0]['output_file']}")
else:
    print("No successful results to combine.")

Combining datasets from multiple tiles...

  Added 100,000 patches from S2A_MSIL2A_20180422T102031_N0500_R065_T3...
  Added 100,000 patches from S2A_MSIL2A_20180604T103021_N0500_R108_T3...
  Added 100,000 patches from S2B_MSIL2A_20180318T102019_N0500_R065_T3...
  Added 100,000 patches from S2B_MSIL2A_20180808T103019_N0500_R108_T3...

✓ Combined dataset saved: /p/scratch/training2600/team1/data/T33VWH/training_data/combined_training_data.npz
  Total patches: 400,000
  Unique classes: 8
  File size: 38.2 MB

Statistics:
  Total patches: 400,000
  Unique classes: 8
  Most common: Class 24 (Coniferous forest) - 270,054 (67.5%)
  Least common: Class 23 (Broad-leaved forest) - 2,777 (0.7%)
  Class imbalance ratio: 97.2:1
